# S6E8 V6 E1 Final: Raw High-Resolution Histogram XGBoost

**Experiment:** does a deliberately *encoding-starved* XGBoost branch, trained on raw
columns only, produce a signal that is (a) strong enough and (b) decorrelated enough
from the current public anchor to be worth a place in a V6 stack? And does raising the
histogram resolution from `max_bin=256` to `max_bin=1024` change anything real?

This notebook is a **two-branch ablation**, not a model search. Everything except
`max_bin` is held identical, including fold assignments, inner early-stopping splits,
and RNG seeds, so that the fold-level deltas form a genuine paired comparison.

## Why E1 uses *fewer* features than V5

V4/V5 feed the model exact-value integer identities, transductive frequency encoding,
fold-safe exact-value target encoding, and reference-distribution statistics from the
external source dataset. Those features hand the model the generator's value lattice
directly.

High-resolution histogram binning is a *second* route to the same structure: with
enough bins, the tree rediscovers the lattice from the raw values on its own. Give a
model both routes and the second one is an expensive duplicate of the first. That is
the most plausible reading of the public result where `max_bin=2047` moved a raw model
by roughly +0.0024 but an already-frequency-encoded model by only about +0.0005.

So E1 withholds the encodings **on purpose**. The goal is not to beat V5 on raw OOF
AUC. The goal is a member whose errors are different from V5's, because in a rank
blend a weaker member still helps whenever

```
d_member > rho * d_anchor        where  d = sqrt(2) * Phi_inverse(AUC)
```

Decorrelation is the cheap lever here; raw strength is the expensive one.

## Honest validation

The V4/V5 code used the **outer validation fold** as the early-stopping `eval_set`.
That chooses the tree count using the labels of the rows being predicted. This
notebook never does that: early stopping runs on an inner 90/10 split carved out of
the outer-training rows, the tree count is recorded, the temporary model is discarded,
and the branch is refit on the *complete* outer-training fold with early stopping off.

## What this notebook will not do

No promise of a Top-10 or Top-3 finish. No automatic public-leaderboard weight search.
No blend file generated without an explicit opt-in. The binormal blend screening at the
end is a transparent sensitivity analysis over assumptions you can see and change, not
an identified optimum.


## Integrated safety revision

This final revision preserves the native-categorical E1 experiment while signing
checkpoints with model parameters, package version, row counts, and train/test ID
fingerprints. Cached fold indices and prediction shapes are validated before reuse,
fold files are written atomically, CPU and GPU checkpoints are kept separate, and
smoke-test predictions cannot be exported as competition submissions.


## 1. Dependencies

`xgboost >= 2.0` is required for `device="cuda"`; `>= 1.6` for native categorical
support. We pin `>= 2.1` because that is where `QuantileDMatrix` plus categorical
features is well settled.

In [ ]:
!pip -q install "kaggle>=1.7" "xgboost>=2.1"
print("dependency install finished")

## 2. Imports and configuration

`FULL_RUN` controls the real experiment. `SMOKE_TEST` overrides it with a small
stratified sample, one partition, and short boosting runs so you can validate the whole
pipeline end to end in a few minutes before committing a T4 session.

In [ ]:
import gc
import hashlib
import json
import math
import os
import subprocess
import sys
import time
import warnings
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb

from scipy.stats import norm, rankdata, spearmanr
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", lambda value: f"{value:,.6f}")

NOTEBOOK_NAME = "s6e8_v6_e1_final_integrated.ipynb"
PIPELINE_VERSION = "v6.e1.final.1"
COMPETITION = "playground-series-s6e8"
TARGET = "addicted_label"
ID_COL = "id"

# ------------------------------------------------------------------ run modes
FULL_RUN = True        # the real experiment
SMOKE_TEST = False     # overrides FULL_RUN with a fast end-to-end validation
USE_GOOGLE_DRIVE = True

EXPECTED_TRAIN_ROWS = 691_369
EXPECTED_TEST_ROWS = 296_302

if SMOKE_TEST:
    PARTITION_SEEDS = [42]
    N_SPLITS = 3
    XGB_MAX_ROUNDS = 300
    EARLY_STOPPING_ROUNDS = 30
    SMOKE_SAMPLE_ROWS = 60_000
else:
    PARTITION_SEEDS = [42, 2024, 7]
    N_SPLITS = 5 if FULL_RUN else 3
    XGB_MAX_ROUNDS = 3000 if FULL_RUN else 500
    EARLY_STOPPING_ROUNDS = 150 if FULL_RUN else 40
    SMOKE_SAMPLE_ROWS = None

MAX_BIN_BRANCHES = [256, 1024]
INNER_VALID_FRACTION = 0.10
INNER_SPLIT_BASE_SEED = 90_001   # deterministic, shared by both branches
MODEL_SEED = 42

# ------------------------------------------------------------------ E1 feature contract
RAW_NUMERIC = [
    "age",
    "daily_screen_time_hours",
    "social_media_hours",
    "gaming_hours",
    "work_study_hours",
    "sleep_hours",
    "notifications_per_day",
    "app_opens_per_day",
    "weekend_screen_time",
]
RAW_CATEGORICAL = [
    "gender",
    "stress_level",
    "academic_work_impact",
]
RAW_COLUMNS = RAW_NUMERIC + RAW_CATEGORICAL

OTHER_SCREEN = "other_screen"
MISSING_CATEGORY_LEVEL = "__MISSING__"

E1_NUMERIC = RAW_NUMERIC + [OTHER_SCREEN]
E1_FEATURES = E1_NUMERIC + RAW_CATEGORICAL

ANCHOR_FILENAME = "submission_s6e8_community_blend.csv"
ANCHOR_PUBLIC_AUC = 0.97128   # displayed public score of the anchor, used for screening only


BASE_XGB_PARAMS = {
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "tree_method": "hist",
    "max_depth": 8,
    "min_child_weight": 64,
    "eta": 0.02,
    "subsample": 0.80,
    "colsample_bytree": 0.70,
    "lambda": 2.0,
    "alpha": 0.0,
    "verbosity": 0,
}

config_for_hash = {
    "pipeline": PIPELINE_VERSION,
    "full_run": FULL_RUN,
    "smoke_test": SMOKE_TEST,
    "partition_seeds": PARTITION_SEEDS,
    "n_splits": N_SPLITS,
    "max_rounds": XGB_MAX_ROUNDS,
    "early_stopping_rounds": EARLY_STOPPING_ROUNDS,
    "inner_valid_fraction": INNER_VALID_FRACTION,
    "inner_split_base_seed": INNER_SPLIT_BASE_SEED,
    "model_seed": MODEL_SEED,
    "max_bin_branches": MAX_BIN_BRANCHES,
    "features": E1_FEATURES,
    "base_xgb_params": BASE_XGB_PARAMS,
    "xgboost_version": xgb.__version__,
    "competition": COMPETITION,
    "expected_train_rows": EXPECTED_TRAIN_ROWS,
    "expected_test_rows": EXPECTED_TEST_ROWS,
}
CONFIG_SIGNATURE = hashlib.sha256(
    json.dumps(config_for_hash, sort_keys=True).encode("utf-8")
).hexdigest()[:12]

xgb_major = int(xgb.__version__.split(".")[0])
if xgb_major < 2:
    raise RuntimeError(
        f"xgboost {xgb.__version__} is too old. This notebook needs >= 2.1 for "
        'device="cuda" together with native categorical features. '
        "Restart the runtime after the install cell."
    )

print(f"xgboost           {xgb.__version__}")
print(f"numpy             {np.__version__}")
print(f"pandas            {pd.__version__}")
print(f"pre-data signature {CONFIG_SIGNATURE}")
print(f"partitions        {PARTITION_SEEDS} x {N_SPLITS} folds")
print(f"branches          max_bin in {MAX_BIN_BRANCHES}")
print(f"smoke test        {SMOKE_TEST}")


## 3. Persistent storage and Kaggle authentication

Everything that takes GPU time is checkpointed to Drive under a directory keyed by the
configuration signature. Re-running this notebook after a Colab disconnect reloads
completed folds instead of retraining them, so the loop is idempotent.

The Kaggle token is read from the Colab Secret `KAGGLE_API_TOKEN` and is never printed
or written to disk by this notebook.

In [ ]:
IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = Path("/kaggle/input").exists()

if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    PERSIST_ROOT = Path("/content/drive/MyDrive/s6e8_v6_e1")
elif IN_KAGGLE:
    PERSIST_ROOT = Path("/kaggle/working/s6e8_v6_e1")
else:
    PERSIST_ROOT = Path("./s6e8_v6_e1")

ANCHOR_DIR = PERSIST_ROOT / "anchors"
ANCHOR_DIR.mkdir(parents=True, exist_ok=True)


def atomic_write_bytes(path: Path, payload: bytes) -> None:
    """Write via a temporary file so a disconnect cannot leave a half-written artifact."""
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_bytes(payload)
    os.replace(tmp, path)


def atomic_savez(path: Path, **arrays) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp.npz")
    np.savez_compressed(tmp, **arrays)
    os.replace(tmp, path)


def atomic_save_npy(path: Path, array: np.ndarray) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp.npy")
    np.save(tmp, array)
    os.replace(tmp, path)


KAGGLE_TOKEN_READY = False
if IN_COLAB:
    from google.colab import userdata

    try:
        _token = userdata.get("KAGGLE_API_TOKEN")
    except Exception as exc:
        raise RuntimeError(
            "Create the KAGGLE_API_TOKEN Colab Secret and enable notebook access for it."
        ) from exc
    if not _token:
        raise RuntimeError("The KAGGLE_API_TOKEN secret exists but is empty.")
    os.environ["KAGGLE_API_TOKEN"] = _token
    del _token
    KAGGLE_TOKEN_READY = True
elif os.environ.get("KAGGLE_API_TOKEN"):
    KAGGLE_TOKEN_READY = True
elif IN_KAGGLE:
    print("Kaggle input mode detected; API authentication may not be required.")

print("persistent root     ", PERSIST_ROOT)
print("kaggle token ready  ", KAGGLE_TOKEN_READY, "(value never printed)")


## 4. Download and validate the competition data

E1 deliberately does **not** download the external source dataset. Reference-distribution
features are part of what E1 withholds.

In [ ]:
if IN_KAGGLE and Path(f"/kaggle/input/{COMPETITION}").exists():
    COMP_DIR = Path(f"/kaggle/input/{COMPETITION}")
else:
    DATA_ROOT = Path("/content/s6e8_v6_data") if IN_COLAB else Path("./s6e8_v6_data")
    COMP_DIR = DATA_ROOT / "competition"
    COMP_DIR.mkdir(parents=True, exist_ok=True)

    required = [COMP_DIR / name for name in ("train.csv", "test.csv", "sample_submission.csv")]
    if not all(path.exists() for path in required):
        if not KAGGLE_TOKEN_READY:
            raise RuntimeError("Kaggle authentication is required to download the data.")
        subprocess.run(
            ["kaggle", "competitions", "download", "-c", COMPETITION, "-p", str(COMP_DIR)],
            check=True,
        )
        for archive_path in COMP_DIR.glob("*.zip"):
            with zipfile.ZipFile(archive_path) as archive:
                archive.extractall(COMP_DIR)

train = pd.read_csv(COMP_DIR / "train.csv")
test = pd.read_csv(COMP_DIR / "test.csv")
sample_submission = pd.read_csv(COMP_DIR / "sample_submission.csv")

# --- structural validation -------------------------------------------------
assert set(RAW_COLUMNS + [ID_COL, TARGET]).issubset(train.columns), "train columns missing"
assert set(RAW_COLUMNS + [ID_COL]).issubset(test.columns), "test columns missing"
assert list(sample_submission.columns) == [ID_COL, TARGET], "unexpected sample_submission schema"
assert train[ID_COL].is_unique, "duplicate train ids"
assert test[ID_COL].is_unique, "duplicate test ids"
assert train[TARGET].notna().all(), "target contains NaN"
assert set(train[TARGET].unique()) <= {0, 1}, "target is not binary"
assert len(sample_submission) == len(test), "sample_submission / test length mismatch"
assert (sample_submission[ID_COL].to_numpy() == test[ID_COL].to_numpy()).all(), (
    "test rows are not in sample_submission order"
)

if FULL_RUN and not SMOKE_TEST:
    assert len(train) == EXPECTED_TRAIN_ROWS, f"expected {EXPECTED_TRAIN_ROWS} train rows"
    assert len(test) == EXPECTED_TEST_ROWS, f"expected {EXPECTED_TEST_ROWS} test rows"

SUBMISSION_IDS = sample_submission[ID_COL].to_numpy()

if SMOKE_TEST and SMOKE_SAMPLE_ROWS is not None:
    train = (
        train.groupby(TARGET, group_keys=False)
        .apply(lambda block: block.sample(
            n=max(1, int(round(SMOKE_SAMPLE_ROWS * len(block) / len(train)))),
            random_state=0,
        ))
        .reset_index(drop=True)
    )
    print(f"SMOKE_TEST: train subsampled to {len(train):,} rows")

y = train[TARGET].to_numpy(dtype=np.int8)


def hash_ids(series: pd.Series) -> str:
    hashed = pd.util.hash_pandas_object(series, index=False).to_numpy(dtype="uint64")
    return hashlib.sha256(hashed.tobytes()).hexdigest()


config_for_hash.update({
    "train_rows": int(len(train)),
    "test_rows": int(len(test)),
    "train_id_hash": hash_ids(train[ID_COL]),
    "test_id_hash": hash_ids(test[ID_COL]),
})
CONFIG_SIGNATURE = hashlib.sha256(
    json.dumps(config_for_hash, sort_keys=True).encode("utf-8")
).hexdigest()[:12]

CHECKPOINT_DIR = PERSIST_ROOT / "checkpoints" / CONFIG_SIGNATURE
ARTIFACT_DIR = PERSIST_ROOT / "artifacts" / CONFIG_SIGNATURE
SUBMISSION_DIR = PERSIST_ROOT / "submissions" / CONFIG_SIGNATURE
for directory in [CHECKPOINT_DIR, ARTIFACT_DIR, SUBMISSION_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"final run signature {CONFIG_SIGNATURE}")
print("checkpoints       ", CHECKPOINT_DIR)
print("artifacts         ", ARTIFACT_DIR)
print("submissions       ", SUBMISSION_DIR)

print(f"train             {train.shape}")
print(f"test              {test.shape}")
print(f"positive rate     {y.mean():.6f}")
print(f"class counts      0: {(y == 0).sum():,}   1: {(y == 1).sum():,}")
print()
print("missingness (fraction NaN, train | test):")
for col in RAW_COLUMNS:
    print(f"  {col:<28} {train[col].isna().mean():.5f} | {test[col].isna().mean():.5f}")


## 5. Build the E1 feature matrix

Twelve raw columns plus exactly one engineered feature. Nothing else.

**On `other_screen` and NaN.** The residual is computed with plain arithmetic, so if any
of its four components is missing the residual is missing too. That is deliberate: NaN
propagation preserves the missingness pattern rather than silently imputing a zero, and
XGBoost learns a default direction for it.

**On categoricals.** One stable target-free representation, shared across train and test.
The vocabulary is built from the union of unlabeled train and test values, so a
test-only level cannot appear as an unseen category at prediction time. Missing values
get an explicit `__MISSING__` level rather than being folded into a default branch,
which keeps "this field was blank" learnable as its own split.

In [ ]:
category_dtypes = {}
for col in RAW_CATEGORICAL:
    combined = pd.concat([train[col], test[col]], ignore_index=True).astype("object")
    combined = combined.where(combined.notna(), MISSING_CATEGORY_LEVEL)
    levels = sorted({str(value) for value in combined.unique()})
    category_dtypes[col] = pd.CategoricalDtype(categories=levels, ordered=False)
    print(f"{col:<28} {len(levels)} levels -> {levels}")


def build_e1_frame(source: pd.DataFrame) -> pd.DataFrame:
    """Return the E1 design matrix. Raw columns only, plus other_screen."""
    frame = pd.DataFrame(index=source.index)

    for col in RAW_NUMERIC:
        frame[col] = source[col].astype("float32")

    frame[OTHER_SCREEN] = (
        source["daily_screen_time_hours"].astype("float32")
        - source["social_media_hours"].astype("float32")
        - source["gaming_hours"].astype("float32")
        - source["work_study_hours"].astype("float32")
    ).astype("float32")

    for col in RAW_CATEGORICAL:
        values = source[col].astype("object")
        values = values.where(source[col].notna(), MISSING_CATEGORY_LEVEL).astype(str)
        frame[col] = values.astype(category_dtypes[col])

    return frame[E1_FEATURES]


X_train = build_e1_frame(train)
X_test = build_e1_frame(test)

# --- assertions on the feature contract ------------------------------------
assert list(X_train.columns) == list(X_test.columns) == E1_FEATURES
assert len(E1_FEATURES) == 13, "E1 must have exactly 13 features"
assert ID_COL not in X_train.columns and TARGET not in X_train.columns
for col in E1_NUMERIC:
    assert X_train[col].dtype == np.float32, f"{col} is not float32"
for col in RAW_CATEGORICAL:
    assert isinstance(X_train[col].dtype, pd.CategoricalDtype)
    assert X_train[col].isna().sum() == 0, "categorical NaN should be an explicit level"
    assert list(X_train[col].cat.categories) == list(X_test[col].cat.categories)

banned_substrings = ("_te", "target_enc", "freq", "code", "cdf", "rank", "pair")
for col in E1_FEATURES:
    assert not any(token in col for token in banned_substrings), f"{col} violates starvation"

train_memory_mb = X_train.memory_usage(deep=True).sum() / 1024**2
test_memory_mb = X_test.memory_usage(deep=True).sum() / 1024**2

print()
print(f"E1 features       {len(E1_FEATURES)}: {E1_FEATURES}")
print(f"X_train           {X_train.shape}  ({train_memory_mb:.1f} MB)")
print(f"X_test            {X_test.shape}  ({test_memory_mb:.1f} MB)")
print(f"other_screen NaN  train {X_train[OTHER_SCREEN].isna().mean():.5f} | "
      f"test {X_test[OTHER_SCREEN].isna().mean():.5f}")
display(X_train.head())

## 6. Pre-flight: can `max_bin=1024` differ from `max_bin=256` at all?

**This is the most important cell in the notebook and it costs nothing to run.**

XGBoost's `hist` builder derives split candidates from a quantile sketch capped at
`max_bin` buckets. If a numeric feature has fewer distinct values than `max_bin`, the
sketch is already effectively lossless and raising `max_bin` cannot add a single new
split point. Synthetic Playground data frequently snaps to a coarse generator lattice,
which is exactly the regime where a resolution ablation is void by construction.

So before spending GPU hours: count distinct values per numeric feature across train
and test. If **no** feature exceeds 256 distinct values, the two branches are provably
the same model and the ablation should be abandoned — though the starved branch itself
is still worth building for its diversity, which is E1's real purpose.

Categorical features are excluded: they are partitioned by category, not binned.

In [ ]:
resolution_rows = []
for col in E1_NUMERIC:
    combined = pd.concat([X_train[col], X_test[col]], ignore_index=True)
    n_unique = int(combined.nunique(dropna=True))
    resolution_rows.append({
        "feature": col,
        "distinct_values": n_unique,
        "nan_fraction": float(combined.isna().mean()),
        "above_256": n_unique > 256,
        "above_1024": n_unique > 1024,
    })

resolution_table = pd.DataFrame(resolution_rows)
display(resolution_table)

n_above_256 = int(resolution_table["above_256"].sum())
n_above_1024 = int(resolution_table["above_1024"].sum())

print()
print(f"numeric features with > 256 distinct values : {n_above_256} of {len(E1_NUMERIC)}")
print(f"numeric features with > 1024 distinct values: {n_above_1024} of {len(E1_NUMERIC)}")
print()

if n_above_256 == 0:
    RESOLUTION_ABLATION_IS_MEANINGFUL = False
    print("PRE-FLIGHT VERDICT: the resolution ablation is VOID.")
    print("  Every numeric feature has fewer than 256 distinct values, so the 256 and")
    print("  1024 sketches contain the same split candidates. Expect the two branches")
    print("  to produce identical (or floating-point-identical) predictions.")
    print("  Recommendation: run the notebook anyway to confirm empirically, then treat")
    print("  max_bin=256 as the E1 branch and spend the remaining budget on E2 instead")
    print("  of on histogram resolution.")
else:
    RESOLUTION_ABLATION_IS_MEANINGFUL = True
    affected = resolution_table.loc[resolution_table["above_256"], "feature"].tolist()
    print("PRE-FLIGHT VERDICT: the resolution ablation is testable.")
    print(f"  Features that can gain split candidates at max_bin=1024: {affected}")
    if n_above_1024 > 0:
        capped = resolution_table.loc[resolution_table["above_1024"], "feature"].tolist()
        print(f"  Features still capped at 1024 bins (some resolution remains unused): {capped}")

atomic_write_bytes(
    ARTIFACT_DIR / "e1_binning_preflight.json",
    json.dumps({
        "features": resolution_rows,
        "n_above_256": n_above_256,
        "n_above_1024": n_above_1024,
        "resolution_ablation_is_meaningful": bool(RESOLUTION_ABLATION_IS_MEANINGFUL),
    }, indent=2).encode("utf-8"),
)

## 7. Freeze three stratified partitions

Fold assignments are generated once, saved as `.npy`, and reloaded on every subsequent
run. Both `max_bin` branches read the same arrays, which is what makes the fold-level
deltas paired rather than two independent noisy measurements.

In [ ]:
fold_assignments = {}

for seed in PARTITION_SEEDS:
    path = ARTIFACT_DIR / f"fold_assignments_{seed}.npy"
    if path.exists():
        folds = np.load(path)
        assert len(folds) == len(train), f"cached folds for seed {seed} have the wrong length"
        print(f"partition {seed:<5} loaded from cache")
    else:
        folds = np.full(len(train), -1, dtype=np.int8)
        splitter = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
        for fold_index, (_, valid_idx) in enumerate(splitter.split(np.zeros(len(train)), y)):
            folds[valid_idx] = fold_index
        assert (folds >= 0).all(), "some rows were not assigned a fold"
        atomic_save_npy(path, folds)
        print(f"partition {seed:<5} created and saved")
    fold_assignments[seed] = folds

summary = []
for seed, folds in fold_assignments.items():
    for fold_index in range(N_SPLITS):
        mask = folds == fold_index
        summary.append({
            "partition": seed,
            "fold": fold_index,
            "rows": int(mask.sum()),
            "positive_rate": float(y[mask].mean()),
        })
display(pd.DataFrame(summary))


## 8. XGBoost specification, with an audit of the proposed parameters

The requested specification was written in scikit-learn style. This notebook uses the
native `xgb.train` API so that `max_bin` can be set on the `QuantileDMatrix` itself and
so that early stopping is an explicit argument rather than a constructor attribute that
must be silently disabled during the refit. The translation is mechanical:

| requested (sklearn) | used (native) | note |
|---|---|---|
| `learning_rate: 0.02` | `eta: 0.02` | alias |
| `reg_lambda: 2.0` | `lambda: 2.0` | canonical name |
| `reg_alpha: 0.0` | `alpha: 0.0` | canonical name |
| `random_state: 42` | `seed: 42` | alias |
| `n_jobs: -1` | `nthread: os.cpu_count()` | native rejects `-1`; Colab free gives 2 vCPUs, and with `device="cuda"` this only affects data marshalling |
| `n_estimators: 3000` | `num_boost_round=3000` | passed to `xgb.train` |
| `early_stopping_rounds: 150` | `xgb.train(..., early_stopping_rounds=150)` | **inner model only** |

**Four substantive audit notes, none of which required changing a value:**

1. `colsample_bytree=0.70` over 13 features means roughly 9 features per tree. That is
   fine and adds useful decorrelation, but it does inject variance into the paired
   comparison. Both branches use `seed=42`, and column sampling is drawn from an RNG
   stream independent of the binning, so the two branches see the **same** column
   subsets tree for tree. The pairing stays tight. Do not vary the seed between
   branches.

2. `eval_metric="auc"` for early stopping is noisier than logloss, because AUC only
   moves when a pair actually swaps order. The 150-round patience is what absorbs that
   noise; it is not generous padding. Keep them together or change neither.

3. `max_depth=8` with `min_child_weight=64` on ~553k outer-training rows is roughly
   256 leaves with a floor of 64 weighted samples each. Reasonable. Depth is doing more
   work than usual here because the feature set is small, so the model has to build
   interactions rather than read them off engineered columns.

4. `max_bin` is set on both the params dict and the `QuantileDMatrix`. XGBoost will
   raise if these disagree, which is a useful guard against a branch silently running
   at the wrong resolution.

The validation and test matrices are plain `DMatrix`, not `QuantileDMatrix`. Prediction
and evaluation use raw feature values and are never binned, so there is nothing to gain
from a quantile sketch there, and it avoids the `ref=` coupling entirely.

In [ ]:
def detect_gpu() -> bool:
    """Confirm GPU training genuinely works rather than trusting a version string."""
    try:
        probe_X = np.random.RandomState(0).rand(64, 3).astype(np.float32)
        probe_y = (probe_X[:, 0] > 0.5).astype(np.int8)
        probe = xgb.DMatrix(probe_X, label=probe_y)
        xgb.train(
            {"objective": "binary:logistic", "tree_method": "hist", "device": "cuda",
             "max_depth": 2, "verbosity": 0},
            probe, num_boost_round=2,
        )
        return True
    except Exception as exc:
        print(f"GPU probe failed: {type(exc).__name__}: {exc}")
        return False


GPU_ACTIVE = detect_gpu()
DEVICE = "cuda" if GPU_ACTIVE else "cpu"

if GPU_ACTIVE:
    try:
        print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                              "--format=csv,noheader"],
                             capture_output=True, text=True, check=True).stdout.strip())
    except Exception:
        pass
    print("GPU training is ACTIVE.")
else:
    print("WARNING: no usable CUDA device. Falling back to CPU.")
    print("         On a 2-vCPU Colab runtime the full experiment will be several times")
    print("         slower. Consider enabling the T4 runtime before committing to it.")


def native_params(max_bin: int, model_seed: int = MODEL_SEED) -> dict:
    params = dict(BASE_XGB_PARAMS)
    params.update({
        "device": DEVICE,
        "max_bin": int(max_bin),
        "seed": int(model_seed),
        "nthread": os.cpu_count() or 2,
    })
    return params

# The test matrix never changes; build it once and reuse it for every fold and branch.
dtest = xgb.DMatrix(X_test, enable_categorical=True)
print(f"\ndtest built: {dtest.num_row():,} rows x {dtest.num_col()} features")
for max_bin in MAX_BIN_BRANCHES:
    print(f"branch max_bin={max_bin}: {native_params(max_bin)}")


## 9. The fold trainer

The contamination-free sequence, per (partition, fold, branch):

1. carve a deterministic stratified 90/10 split out of the **outer-training** rows;
2. train a temporary model with early stopping on the inner validation labels only;
3. record `best_iteration + 1`;
4. discard the temporary model;
5. refit on the **complete** outer-training fold, early stopping off, `num_boost_round`
   fixed to the recorded count;
6. predict the untouched outer validation fold and the full test set;
7. checkpoint atomically to Drive.

The inner split seed depends on the partition and the fold but **not** on `max_bin`, so
both branches early-stop against the identical inner validation rows.

In [ ]:
def inner_split_seed(partition_seed: int, fold_index: int) -> int:
    """Deterministic and independent of max_bin, so both branches share the inner split."""
    return INNER_SPLIT_BASE_SEED + 1000 * int(partition_seed) + int(fold_index)


def model_seed(partition_seed: int, fold_index: int) -> int:
    """Change stochastic draws across folds while preserving branch pairing."""
    return MODEL_SEED + 10_000 * int(partition_seed) + int(fold_index)


def checkpoint_path(partition_seed: int, fold_index: int, max_bin: int) -> Path:
    device_tag = DEVICE.replace(":", "_")
    return CHECKPOINT_DIR / (
        f"e1_p{partition_seed}_f{fold_index}_mb{max_bin}_{device_tag}.npz"
    )


def train_one_fold(partition_seed: int, fold_index: int, max_bin: int,
                   num_boost_round: int = None, persist: bool = True) -> dict:
    """Train one paired outer-fold unit with inner-only early stopping."""
    folds = fold_assignments[partition_seed]
    valid_mask = folds == fold_index
    train_idx = np.flatnonzero(~valid_mask)
    valid_idx = np.flatnonzero(valid_mask).astype(np.int32)
    path = checkpoint_path(partition_seed, fold_index, max_bin)

    if persist and path.exists():
        with np.load(path, allow_pickle=False) as cached:
            cached_valid_idx = cached["valid_idx"].astype(np.int32)
            if not np.array_equal(cached_valid_idx, valid_idx):
                raise RuntimeError(f"Checkpoint fold-index mismatch: {path}")
            valid_pred = cached["valid_pred"].astype(np.float32)
            test_pred = cached["test_pred"].astype(np.float32)
            if valid_pred.shape != (len(valid_idx),) or test_pred.shape != (len(test),):
                raise RuntimeError(f"Checkpoint prediction-shape mismatch: {path}")
            if not np.isfinite(valid_pred).all() or not np.isfinite(test_pred).all():
                raise RuntimeError(f"Checkpoint contains non-finite predictions: {path}")
            return {
                "valid_idx": cached_valid_idx,
                "valid_pred": valid_pred,
                "test_pred": test_pred,
                "best_rounds": int(cached["best_rounds"].item()),
                "fit_seconds": float(cached["fit_seconds"].item()),
                "valid_auc": float(cached["valid_auc"].item()),
                "hit_ceiling": bool(cached["hit_ceiling"].item()),
                "from_cache": True,
            }

    X_outer = X_train.iloc[train_idx]
    y_outer = y[train_idx]
    started = time.time()
    params = native_params(max_bin, model_seed(partition_seed, fold_index))

    if num_boost_round is None:
        fit_idx, inner_valid_idx = train_test_split(
            np.arange(len(train_idx)),
            test_size=INNER_VALID_FRACTION,
            stratify=y_outer,
            random_state=inner_split_seed(partition_seed, fold_index),
            shuffle=True,
        )
        d_fit = xgb.QuantileDMatrix(
            X_outer.iloc[fit_idx], label=y_outer[fit_idx],
            max_bin=max_bin, enable_categorical=True,
        )
        d_inner_valid = xgb.DMatrix(
            X_outer.iloc[inner_valid_idx], label=y_outer[inner_valid_idx],
            enable_categorical=True,
        )
        probe = xgb.train(
            params,
            d_fit,
            num_boost_round=XGB_MAX_ROUNDS,
            evals=[(d_inner_valid, "inner_valid")],
            early_stopping_rounds=EARLY_STOPPING_ROUNDS,
            verbose_eval=False,
        )
        best_rounds = int(probe.best_iteration) + 1
        hit_ceiling = best_rounds >= XGB_MAX_ROUNDS
        del probe, d_fit, d_inner_valid
        gc.collect()
    else:
        best_rounds = int(num_boost_round)
        hit_ceiling = False

    d_outer = xgb.QuantileDMatrix(
        X_outer, label=y_outer, max_bin=max_bin, enable_categorical=True,
    )
    booster = xgb.train(params, d_outer, num_boost_round=best_rounds, verbose_eval=False)
    d_valid = xgb.DMatrix(X_train.iloc[valid_idx], enable_categorical=True)
    valid_pred = booster.predict(d_valid).astype(np.float32)
    test_pred = booster.predict(dtest).astype(np.float32)

    del booster, d_outer, d_valid, X_outer
    gc.collect()

    fit_seconds = time.time() - started
    valid_auc = float(roc_auc_score(y[valid_idx], valid_pred))
    assert valid_pred.shape == (len(valid_idx),)
    assert test_pred.shape == (len(test),)
    assert np.isfinite(valid_pred).all() and np.isfinite(test_pred).all()

    result = {
        "valid_idx": valid_idx,
        "valid_pred": valid_pred,
        "test_pred": test_pred,
        "best_rounds": best_rounds,
        "fit_seconds": fit_seconds,
        "valid_auc": valid_auc,
        "hit_ceiling": hit_ceiling,
        "from_cache": False,
    }
    if persist:
        atomic_savez(
            path,
            valid_idx=valid_idx,
            valid_pred=valid_pred,
            test_pred=test_pred,
            best_rounds=np.int32(best_rounds),
            fit_seconds=np.float64(fit_seconds),
            valid_auc=np.float64(valid_auc),
            hit_ceiling=np.bool_(hit_ceiling),
        )
    return result


print("fold trainer ready")
print(f"checkpoints already present: "
      f"{len(list(CHECKPOINT_DIR.glob('e1_p*_f*_mb*.npz')))} / "
      f"{len(PARTITION_SEEDS) * N_SPLITS * len(MAX_BIN_BRANCHES)}")


## 10. Honest runtime estimate before committing the session

Rather than quoting a guess, this cell trains a short fixed-round model on one fold per
branch, measures it, and extrapolates. The extrapolation assumes cost scales roughly
linearly in boosting rounds and that early stopping lands somewhere between 40% and
100% of the ceiling, so a range is reported rather than a single number.

In [ ]:
CALIBRATION_ROUNDS = 100

calibration = {}
for max_bin in MAX_BIN_BRANCHES:
    started = time.time()
    _ = train_one_fold(PARTITION_SEEDS[0], 0, max_bin,
                       num_boost_round=CALIBRATION_ROUNDS, persist=False)
    elapsed = time.time() - started
    per_round = elapsed / CALIBRATION_ROUNDS
    calibration[max_bin] = per_round
    print(f"max_bin={max_bin:<5} {elapsed:6.1f}s for {CALIBRATION_ROUNDS} rounds "
          f"-> {per_round * 1000:6.1f} ms/round")

n_units = len(PARTITION_SEEDS) * N_SPLITS
# each unit trains the probe (up to XGB_MAX_ROUNDS) and then refits (best_rounds rounds)
low_hours = 0.0
high_hours = 0.0
for max_bin, per_round in calibration.items():
    low_rounds = n_units * (0.40 * XGB_MAX_ROUNDS + 0.40 * XGB_MAX_ROUNDS)
    high_rounds = n_units * (1.00 * XGB_MAX_ROUNDS + 1.00 * XGB_MAX_ROUNDS)
    low_hours += low_rounds * per_round / 3600
    high_hours += high_rounds * per_round / 3600

done = len(list(CHECKPOINT_DIR.glob("e1_p*_f*_mb*.npz")))
total_units = n_units * len(MAX_BIN_BRANCHES)
remaining_fraction = max(0.0, 1.0 - done / total_units)

print()
print(f"units to train        {total_units}  ({done} already checkpointed)")
print(f"ESTIMATED TOTAL       {low_hours:.2f} - {high_hours:.2f} hours on this runtime")
print(f"ESTIMATED REMAINING   {low_hours * remaining_fraction:.2f} - "
      f"{high_hours * remaining_fraction:.2f} hours")
print()
print("The upper bound assumes early stopping never triggers before the ceiling, which")
print("is pessimistic at eta=0.02 with 150-round patience. Expect the lower half of the")
print("range. Checkpointing means a disconnect costs at most one fold.")

## 11. Run the experiment

Safe to re-run after a disconnect: completed units are reloaded from Drive.

In [ ]:
oof_predictions = {
    max_bin: {seed: np.full(len(train), np.nan, dtype=np.float32) for seed in PARTITION_SEEDS}
    for max_bin in MAX_BIN_BRANCHES
}
test_predictions = {
    max_bin: {seed: np.zeros(len(test), dtype=np.float64) for seed in PARTITION_SEEDS}
    for max_bin in MAX_BIN_BRANCHES
}
fold_records = []

run_started = time.time()
for partition_seed in PARTITION_SEEDS:
    for fold_index in range(N_SPLITS):
        for max_bin in MAX_BIN_BRANCHES:
            result = train_one_fold(partition_seed, fold_index, max_bin)

            oof_predictions[max_bin][partition_seed][result["valid_idx"]] = result["valid_pred"]
            test_predictions[max_bin][partition_seed] += result["test_pred"] / N_SPLITS

            fold_records.append({
                "partition": partition_seed,
                "fold": fold_index,
                "max_bin": max_bin,
                "valid_auc": result["valid_auc"],
                "best_rounds": result["best_rounds"],
                "fit_seconds": result["fit_seconds"],
                "from_cache": result["from_cache"],
                "n_valid": len(result["valid_idx"]),
            })

            tag = "cache" if result["from_cache"] else f"{result['fit_seconds']:6.1f}s"
            print(f"p{partition_seed:<5} fold {fold_index}  max_bin={max_bin:<5} "
                  f"AUC {result['valid_auc']:.6f}  rounds {result['best_rounds']:>5}  {tag}")
        gc.collect()

for max_bin in MAX_BIN_BRANCHES:
    for seed in PARTITION_SEEDS:
        assert np.isfinite(oof_predictions[max_bin][seed]).all(), (
            f"OOF has gaps for max_bin={max_bin}, partition={seed}"
        )

fold_metrics = pd.DataFrame(fold_records)
fold_metrics.to_csv(ARTIFACT_DIR / "e1_fold_metrics.csv", index=False)

print(f"\nwall clock this session: {(time.time() - run_started) / 60:.1f} min")
display(fold_metrics.groupby("max_bin")[["valid_auc", "best_rounds", "fit_seconds"]].agg(
    ["mean", "std", "min", "max"]
))

if (fold_metrics["best_rounds"] >= XGB_MAX_ROUNDS).any():
    print("\nWARNING: at least one fold hit the boosting ceiling. Early stopping never")
    print("         triggered there, so the tree count is a cap rather than a choice.")
    print(f"         Consider raising XGB_MAX_ROUNDS above {XGB_MAX_ROUNDS}.")

## 12. Paired ablation report

The requested GO rule is implemented literally, and then audited.

**The audit.** The 15 fold deltas are not 15 independent observations. Folds *within* a
partition are disjoint, but the three partitions re-slice the **same 691,369 rows**.
A one-sample t-test on 15 correlated deltas therefore overstates its own significance,
and the sign test's nominal p-value (13/15 positive gives p ~ 0.007 under independence)
is optimistic for the same reason.

The three **partition-level** pooled deltas are the closer thing to independent replicates
here, and even they share the underlying data. So the honest reading is:

- treat the t-statistic as descriptive, not inferential;
- weight the sign-consistency evidence more heavily than its p-value;
- require all three partition-level deltas to agree in sign, which is the criterion the
  correlation structure damages least.

A real effect is consistently positive. Selection noise is not.

In [ ]:
def pooled_auc(pred_vector: np.ndarray) -> float:
    return float(roc_auc_score(y, pred_vector))


partition_pooled = {
    max_bin: {seed: pooled_auc(oof_predictions[max_bin][seed]) for seed in PARTITION_SEEDS}
    for max_bin in MAX_BIN_BRANCHES
}

print("Pooled OOF AUC by partition")
print(f"{'partition':<12}" + "".join(f"max_bin={mb:<10}" for mb in MAX_BIN_BRANCHES) + "delta")
for seed in PARTITION_SEEDS:
    values = [partition_pooled[mb][seed] for mb in MAX_BIN_BRANCHES]
    print(f"{seed:<12}" + "".join(f"{v:<18.6f}" for v in values)
          + f"{values[-1] - values[0]:+.6f}")

partition_deltas = np.array([
    partition_pooled[1024][seed] - partition_pooled[256][seed] for seed in PARTITION_SEEDS
])

wide = fold_metrics.pivot_table(index=["partition", "fold"], columns="max_bin",
                                values="valid_auc")
wide["delta_1024_minus_256"] = wide[1024] - wide[256]
display(wide)

deltas = wide["delta_1024_minus_256"].to_numpy()
n_positive = int((deltas > 0).sum())
delta_mean = float(deltas.mean())
delta_sd = float(deltas.std(ddof=1)) if len(deltas) > 1 else 0.0
delta_se = delta_sd / math.sqrt(len(deltas)) if delta_sd > 0 else float("nan")
t_stat = delta_mean / delta_se if delta_se and np.isfinite(delta_se) and delta_se > 0 else float("nan")

# Spearman between the two branches, per partition and on the partition mean
oof_mean = {
    mb: np.mean([oof_predictions[mb][s] for s in PARTITION_SEEDS], axis=0).astype(np.float32)
    for mb in MAX_BIN_BRANCHES
}
branch_spearman = float(spearmanr(oof_mean[256], oof_mean[1024]).statistic)
max_abs_diff = float(np.abs(oof_mean[256] - oof_mean[1024]).max())

print()
print(f"fold deltas (1024 - 256): {n_positive} of {len(deltas)} positive")
print(f"  mean   {delta_mean:+.7f}")
print(f"  sd     {delta_sd:.7f}")
print(f"  se     {delta_se:.7f}")
print(f"  t      {t_stat:+.3f}   (descriptive only; the 15 deltas are correlated)")
print(f"partition-level deltas    : {np.array2string(partition_deltas, precision=7)}")
print(f"all partition deltas > 0  : {bool((partition_deltas > 0).all())}")
print()
print(f"Spearman(OOF_256, OOF_1024)      {branch_spearman:.8f}")
print(f"max |prob difference|            {max_abs_diff:.3e}")

# --- is this comparison measuring binning, or early-stopping noise? ----------
rounds_wide = fold_metrics.pivot_table(index=["partition", "fold"], columns="max_bin",
                                       values="best_rounds")
rounds_ratio = (rounds_wide[1024] / rounds_wide[256].clip(lower=1)).to_numpy()
print()
print(f"best_rounds 1024/256 ratio       median {np.median(rounds_ratio):.3f}   "
      f"range [{rounds_ratio.min():.3f}, {rounds_ratio.max():.3f}]")
if rounds_ratio.min() < 0.5 or rounds_ratio.max() > 2.0:
    print("  WARNING: the two branches are stopping at very different tree counts on at")
    print("  least one fold. The inner validation split is only "
          f"{INNER_VALID_FRACTION:.0%} of the outer-training")
    print("  rows, and AUC-based early stopping is noisy on small validation sets. When")
    print("  the counts diverge this much, the fold delta is partly measuring")
    print("  early-stopping variance rather than histogram resolution. Before trusting")
    print("  the GO decision, re-run the affected folds with num_boost_round pinned to a")
    print("  single shared value (the median best_rounds) for both branches.")

if max_abs_diff < 1e-9:
    print("\nThe two branches are numerically IDENTICAL. This confirms the section 6")
    print("pre-flight: max_bin was never the binding constraint. Report the ablation as")
    print("void rather than as a null result -- there was no treatment to test.")

rule_all_partitions = bool((partition_deltas > 0).all())
rule_sign = n_positive >= 13 if len(deltas) == 15 else n_positive >= math.ceil(0.85 * len(deltas))
rule_t = bool(np.isfinite(t_stat) and t_stat > 2.5)
GO_1024 = rule_all_partitions and rule_sign and rule_t

print()
print("Requested GO rule")
print(f"  all partition deltas positive : {rule_all_partitions}")
print(f"  >= 13 of 15 fold deltas > 0   : {rule_sign}  ({n_positive}/{len(deltas)})")
print(f"  paired t > 2.5                : {rule_t}  (t = {t_stat:+.3f})")
print(f"  ==> GO for max_bin=1024       : {GO_1024}")

SELECTED_MAX_BIN = 1024 if GO_1024 else 256
print()
print(f"E1 branch selected for downstream use: max_bin={SELECTED_MAX_BIN}")
if not GO_1024:
    print("  max_bin=256 is retained. Its value to V6 is decorrelation from V5, which is")
    print("  measured in the next section and is independent of this ablation's outcome.")

ablation_report = {
    "pipeline_version": PIPELINE_VERSION,
    "config_signature": CONFIG_SIGNATURE,
    "partition_pooled_auc": {str(mb): {str(s): partition_pooled[mb][s]
                                       for s in PARTITION_SEEDS}
                             for mb in MAX_BIN_BRANCHES},
    "partition_deltas": partition_deltas.tolist(),
    "fold_deltas": deltas.tolist(),
    "n_fold_deltas_positive": n_positive,
    "n_fold_deltas": int(len(deltas)),
    "delta_mean": delta_mean,
    "delta_sd": delta_sd,
    "delta_se": delta_se,
    "paired_t_statistic": None if not np.isfinite(t_stat) else t_stat,
    "spearman_between_branches": branch_spearman,
    "best_rounds_ratio_median": float(np.median(rounds_ratio)),
    "best_rounds_ratio_min": float(rounds_ratio.min()),
    "best_rounds_ratio_max": float(rounds_ratio.max()),
    "max_abs_probability_difference": max_abs_diff,
    "rule_all_partition_deltas_positive": rule_all_partitions,
    "rule_sign_consistency": bool(rule_sign),
    "rule_t_above_2p5": rule_t,
    "go_for_max_bin_1024": bool(GO_1024),
    "selected_max_bin": int(SELECTED_MAX_BIN),
    "caveat": (
        "The 15 fold deltas re-slice the same training rows across three partitions and "
        "are therefore correlated. The t-statistic is descriptive, not inferential."
    ),
}
atomic_write_bytes(ARTIFACT_DIR / "e1_paired_ablation_report.json",
                   json.dumps(ablation_report, indent=2).encode("utf-8"))

## 13. Export OOF and test arrays

Test predictions: averaged over the five folds within a partition, then averaged across
partitions. Kept in `float32`.

In [ ]:
final_oof = {mb: oof_mean[mb] for mb in MAX_BIN_BRANCHES}
final_test = {
    mb: np.mean([test_predictions[mb][s] for s in PARTITION_SEEDS], axis=0).astype(np.float32)
    for mb in MAX_BIN_BRANCHES
}

for mb in MAX_BIN_BRANCHES:
    assert np.isfinite(final_oof[mb]).all() and np.isfinite(final_test[mb]).all()
    assert len(final_test[mb]) == len(test)
    np.save(ARTIFACT_DIR / f"oof_e1_maxbin{mb}.npy", final_oof[mb])
    np.save(ARTIFACT_DIR / f"test_e1_maxbin{mb}.npy", final_test[mb])
    print(f"max_bin={mb:<5} partition-mean OOF AUC {pooled_auc(final_oof[mb]):.6f}   "
          f"test pred range [{final_test[mb].min():.6f}, {final_test[mb].max():.6f}]")

E1_OOF_AUC = {mb: pooled_auc(final_oof[mb]) for mb in MAX_BIN_BRANCHES}
print(f"\nnote: the partition-mean OOF is itself a 3-model average, so it sits slightly")
print(f"      above any single partition's pooled OOF. Use the single-partition numbers")
print(f"      in section 12 when comparing against V5's reported OOF.")

## 14. Anchor diversity analysis

Place `submission_s6e8_community_blend.csv` in the anchors directory on Drive, or upload
it when prompted.

Raw Pearson correlation against the anchor is deliberately **not** reported. The anchor
is already a uniform percentile rank (its values run from `0.5/n` to `1 - 0.5/n`) while
XGBoost emits calibrated probabilities, so a raw Pearson value measures the difference
in calibration, not the difference in signal. ROC AUC depends only on ordering.

In [ ]:
anchor_path = ANCHOR_DIR / ANCHOR_FILENAME
if not anchor_path.exists() and IN_COLAB:
    print(f"{ANCHOR_FILENAME} not found in {ANCHOR_DIR}. Upload it now, or skip.")
    from google.colab import files

    uploaded = files.upload()
    for name, payload in uploaded.items():
        atomic_write_bytes(ANCHOR_DIR / ANCHOR_FILENAME, payload)
        print(f"saved {name} as {ANCHOR_FILENAME}")

ANCHOR_AVAILABLE = anchor_path.exists()
anchor_report = {"available": ANCHOR_AVAILABLE}


def percentile_rank(values: np.ndarray) -> np.ndarray:
    return (rankdata(values, method="average") - 0.5) / len(values)


def extreme_overlap(a_rank: np.ndarray, b_rank: np.ndarray, fraction: float, top: bool):
    k = max(1, int(round(fraction * len(a_rank))))
    order_a = np.argsort(-a_rank if top else a_rank, kind="stable")[:k]
    order_b = np.argsort(-b_rank if top else b_rank, kind="stable")[:k]
    return len(np.intersect1d(order_a, order_b)) / k


if ANCHOR_AVAILABLE:
    anchor = pd.read_csv(anchor_path)
    assert list(anchor.columns) == [ID_COL, TARGET], "unexpected anchor schema"
    assert len(anchor) == len(sample_submission), "anchor row count mismatch"
    assert (anchor[ID_COL].to_numpy() == SUBMISSION_IDS).all(), "anchor id order mismatch"
    assert np.isfinite(anchor[TARGET].to_numpy()).all(), "anchor has non-finite values"

    anchor_rank = percentile_rank(anchor[TARGET].to_numpy())

    for mb in MAX_BIN_BRANCHES:
        member_rank = percentile_rank(final_test[mb])
        displacement = np.abs(member_rank - anchor_rank)
        rho = float(spearmanr(final_test[mb], anchor[TARGET].to_numpy()).statistic)

        entry = {
            "spearman": rho,
            "mean_abs_rank_displacement": float(displacement.mean()),
            "median_abs_rank_displacement": float(np.median(displacement)),
            "p95_abs_rank_displacement": float(np.percentile(displacement, 95)),
            "max_abs_rank_displacement": float(displacement.max()),
        }
        for fraction in (0.01, 0.05, 0.10):
            entry[f"top_{int(fraction * 100)}pct_overlap"] = extreme_overlap(
                member_rank, anchor_rank, fraction, top=True)
            entry[f"bottom_{int(fraction * 100)}pct_overlap"] = extreme_overlap(
                member_rank, anchor_rank, fraction, top=False)
        anchor_report[f"maxbin{mb}"] = entry

        print(f"\n--- E1 max_bin={mb} versus anchor ---")
        print(f"  Spearman                      {rho:.8f}")
        print(f"  mean |rank displacement|      {entry['mean_abs_rank_displacement']:.6f}")
        print(f"  median |rank displacement|    {entry['median_abs_rank_displacement']:.6f}")
        print(f"  p95 |rank displacement|       {entry['p95_abs_rank_displacement']:.6f}")
        print(f"  max |rank displacement|       {entry['max_abs_rank_displacement']:.6f}")
        for fraction in (1, 5, 10):
            print(f"  top {fraction:>2}% overlap             "
                  f"{entry[f'top_{fraction}pct_overlap']:.4f}"
                  f"    bottom {fraction:>2}% overlap  "
                  f"{entry[f'bottom_{fraction}pct_overlap']:.4f}")

    print("\nFor reference, V5 measured Spearman 0.99480 against this anchor. E1 is more")
    print("useful to a blend than V5 only if it is materially less correlated than that.")
else:
    print("Anchor not available. The blend screening below will run on assumed rho values.")

atomic_write_bytes(ARTIFACT_DIR / "e1_anchor_correlation_report.json",
                   json.dumps(anchor_report, indent=2).encode("utf-8"))

## 15. Binormal blend screening

Under an equal-variance binormal model, a score's separation is
`d = sqrt(2) * Phi_inverse(AUC)`, and a rank blend `w * member + (1 - w) * anchor` has

```
d_blend(w) = (w*d_m + (1-w)*d_a) / sqrt(1 - 2*(1-rho)*w*(1-w))
```

Maximising that gives the closed form

```
w* = (d_m - rho*d_a) / ((1 - rho) * (d_a + d_m))
```

so the member helps at all only when `d_m > rho * d_a`. Note what this means: a member
that is *worse* than the anchor can still be worth a positive weight, provided its
correlation is low enough. "It scores lower, so don't blend it" is not a valid rule.

### Three limitations, stated plainly

1. **`rho` is the wrong correlation.** The binormal derivation needs the standardized
   *within-class* score correlation. What we can measure without test labels is the
   unconditional Spearman correlation over all test rows. These differ, usually with
   the unconditional value biased upward because the class signal itself is shared.
   Treat `rho` as a screening proxy, not as an identified parameter.

2. **`d_m` is unknown.** We have the member's OOF AUC, not its public AUC. V5's single
   observed OOF-to-public offset was about +0.00106, but one observation is a prior,
   not a calibration constant. The table below therefore sweeps several offsets rather
   than committing to one.

3. **`w*` is not statistically identified.** No confidence interval is computed and none
   should be implied. The table shows how the answer moves with the assumptions; it does
   not tell you which assumption is true.

The proper fix is to reconstruct aligned OOF predictions for the anchor and evaluate
blend weights against labels with nested OOF or a paired DeLong/bootstrap analysis. This
screening is what is available in the absence of that.

In [ ]:
SQRT2 = math.sqrt(2.0)


def d_from_auc(auc: float) -> float:
    return SQRT2 * norm.ppf(auc)


def auc_from_d(d_value: float) -> float:
    return float(norm.cdf(d_value / SQRT2))


def blend_auc(d_member: float, d_anchor: float, rho: float, weight: float) -> float:
    numerator = weight * d_member + (1.0 - weight) * d_anchor
    denominator = math.sqrt(max(1.0 - 2.0 * (1.0 - rho) * weight * (1.0 - weight), 1e-12))
    return auc_from_d(numerator / denominator)


def unconstrained_w_star(d_member: float, d_anchor: float, rho: float) -> float:
    if rho >= 1.0:
        return float("-inf") if d_member < d_anchor else float("inf")
    return (d_member - rho * d_anchor) / ((1.0 - rho) * (d_anchor + d_member))


D_ANCHOR = d_from_auc(ANCHOR_PUBLIC_AUC)
OFFSETS = [0.0000, 0.0005, 0.0010, 0.0015]

rows = []
for mb in MAX_BIN_BRANCHES:
    if ANCHOR_AVAILABLE:
        rho_values = [("measured", anchor_report[f"maxbin{mb}"]["spearman"])]
    else:
        rho_values = [("assumed", value) for value in (0.99, 0.98, 0.97)]

    for rho_label, rho in rho_values:
        for offset in OFFSETS:
            assumed_auc = min(E1_OOF_AUC[mb] + offset, 0.999999)
            d_member = d_from_auc(assumed_auc)
            break_even_auc = auc_from_d(rho * D_ANCHOR)
            w_raw = unconstrained_w_star(d_member, D_ANCHOR, rho)
            w_clipped = float(min(max(w_raw, 0.0), 1.0))
            rows.append({
                "max_bin": mb,
                "rho_source": rho_label,
                "rho": rho,
                "oof_offset": offset,
                "assumed_member_auc": assumed_auc,
                "break_even_auc": break_even_auc,
                "clears_break_even": assumed_auc > break_even_auc,
                "w_star_unconstrained": w_raw,
                "w_star_clipped": w_clipped,
                "blended_auc": blend_auc(d_member, D_ANCHOR, rho, w_clipped),
                "gain_vs_anchor": blend_auc(d_member, D_ANCHOR, rho, w_clipped) - ANCHOR_PUBLIC_AUC,
            })

sensitivity = pd.DataFrame(rows)
pd.set_option("display.float_format", lambda value: f"{value:,.6f}")
display(sensitivity)
sensitivity.to_csv(ARTIFACT_DIR / "e1_blend_sensitivity.csv", index=False)

print(f"anchor public AUC {ANCHOR_PUBLIC_AUC:.5f}  ->  d_anchor = {D_ANCHOR:.6f}")
print()
any_clears = bool(sensitivity["clears_break_even"].any())
if any_clears:
    best = sensitivity.loc[sensitivity["gain_vs_anchor"].idxmax()]
    print("At least one assumption clears break-even. Most favourable row:")
    print(f"  max_bin={int(best['max_bin'])}  rho={best['rho']:.6f}  "
          f"offset={best['oof_offset']:.4f}")
    print(f"  w* = {best['w_star_clipped']:.4f}  ->  modelled blended AUC "
          f"{best['blended_auc']:.6f}  ({best['gain_vs_anchor']:+.6f})")
    print("  This is the MOST optimistic assumption in the sweep, not an expectation.")
else:
    print("No assumption in the sweep clears break-even. Under this screening, E1 does")
    print("not currently justify an anchor blend. Its value would have to come from a")
    print("multi-member V6 stack where the correlation structure is measured against")
    print("labels rather than assumed.")

print()
print("Reminder: rho here is unconditional Spearman on unlabeled test rows, which is a")
print("proxy for the within-class correlation the model actually requires. No confidence")
print("interval is implied for w*.")

## 16. Build the two self-trained submissions

Both branches are always exported, regardless of the ablation outcome, so the artifacts
are complete and the decision remains auditable after the fact.

In [ ]:
def build_submission(prediction: np.ndarray, filename: str) -> Path:
    if SMOKE_TEST:
        raise RuntimeError("SMOKE_TEST outputs cannot be exported as submissions.")
    assert len(prediction) == len(sample_submission), "prediction length mismatch"
    assert np.isfinite(prediction).all(), "non-finite predictions"

    frame = pd.DataFrame({ID_COL: SUBMISSION_IDS, TARGET: prediction.astype(np.float64)})
    assert (frame[ID_COL].to_numpy() == SUBMISSION_IDS).all(), "submission id order changed"
    assert len(frame) == EXPECTED_TEST_ROWS or SMOKE_TEST, (
        f"expected {EXPECTED_TEST_ROWS} submission rows, got {len(frame)}"
    )
    assert frame[TARGET].nunique() > 1000, "suspiciously low prediction cardinality"

    drive_path = SUBMISSION_DIR / filename
    frame.to_csv(drive_path, index=False)
    digest = hashlib.sha256(drive_path.read_bytes()).hexdigest()

    local_path = Path("/content") / filename
    if IN_COLAB:
        local_path.write_bytes(drive_path.read_bytes())

    print(f"{filename}")
    print(f"  rows        {len(frame):,}")
    print(f"  range       [{frame[TARGET].min():.8f}, {frame[TARGET].max():.8f}]")
    print(f"  distinct    {frame[TARGET].nunique():,}")
    print(f"  sha256      {digest}")
    print(f"  drive       {drive_path}")
    if IN_COLAB:
        print(f"  local       {local_path}")
    return drive_path


submission_hashes = {}
if SMOKE_TEST:
    print("SMOKE_TEST is active: no official-looking submission files were created.")
else:
    for mb in MAX_BIN_BRANCHES:
        filename = f"submission_s6e8_v6_e1_maxbin{mb}.csv"
        path = build_submission(final_test[mb], filename)
        submission_hashes[filename] = hashlib.sha256(path.read_bytes()).hexdigest()
        print()


## 17. Optional anchor blend (disabled by default)

**Do not enable this cell before reading the section 15 table.** Set `ENABLE_ANCHOR_BLEND
= True` and choose `BLEND_WEIGHT` yourself. The notebook will not pick a weight for you,
and it will refuse to run if the chosen weight fails the break-even screen at the
measured correlation, unless you also set `OVERRIDE_BREAK_EVEN = True`.

In [ ]:
ENABLE_ANCHOR_BLEND = False        # <- deliberate opt-in
BLEND_WEIGHT = None                # <- e.g. 0.15; you choose it, not the notebook
BLEND_MAX_BIN = None               # <- e.g. 256
OVERRIDE_BREAK_EVEN = False
ASSUMED_OFFSET_FOR_CHECK = 0.0010  # the OOF -> public offset you are willing to assume

if not ENABLE_ANCHOR_BLEND:
    print("Anchor blend disabled. Review the section 15 sensitivity table first.")
    print("To enable: set ENABLE_ANCHOR_BLEND=True, BLEND_WEIGHT=<float>, "
          "BLEND_MAX_BIN=<256|1024>.")
else:
    if not ANCHOR_AVAILABLE:
        raise RuntimeError("The anchor file is required for a blend.")
    if BLEND_WEIGHT is None or BLEND_MAX_BIN is None:
        raise ValueError("Set both BLEND_WEIGHT and BLEND_MAX_BIN explicitly.")
    if not 0.0 < BLEND_WEIGHT < 1.0:
        raise ValueError("BLEND_WEIGHT must lie strictly between 0 and 1.")

    rho = anchor_report[f"maxbin{BLEND_MAX_BIN}"]["spearman"]
    assumed = min(E1_OOF_AUC[BLEND_MAX_BIN] + ASSUMED_OFFSET_FOR_CHECK, 0.999999)
    break_even = auc_from_d(rho * D_ANCHOR)

    print(f"rho                {rho:.8f}")
    print(f"assumed member AUC {assumed:.6f}   break-even {break_even:.6f}")

    if assumed <= break_even and not OVERRIDE_BREAK_EVEN:
        raise RuntimeError(
            f"Refusing to build the blend: the assumed member AUC {assumed:.6f} does not "
            f"clear the break-even threshold {break_even:.6f} at rho={rho:.6f}. "
            "Set OVERRIDE_BREAK_EVEN=True only if you have a documented reason."
        )

    member_rank = percentile_rank(final_test[BLEND_MAX_BIN])
    anchor_rank_local = percentile_rank(anchor[TARGET].to_numpy())
    blended = BLEND_WEIGHT * member_rank + (1.0 - BLEND_WEIGHT) * anchor_rank_local

    weight_tag = f"{int(round(BLEND_WEIGHT * 100)):02d}"
    filename = f"submission_s6e8_v6_e1_anchor_self{weight_tag}_mb{BLEND_MAX_BIN}.csv"
    path = build_submission(blended, filename)
    submission_hashes[filename] = hashlib.sha256(path.read_bytes()).hexdigest()

    print(f"\nSpearman(blend, anchor) "
          f"{spearmanr(blended, anchor[TARGET].to_numpy()).statistic:.8f}")
    print("This blend is a precommitted experiment, not a validated expected gain.")

## 18. Run manifest

In [ ]:
manifest = {
    "notebook": NOTEBOOK_NAME,
    "pipeline_version": PIPELINE_VERSION,
    "config_signature": CONFIG_SIGNATURE,
    "generated_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "environment": {
        "python": sys.version.split()[0],
        "xgboost": xgb.__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "gpu_active": bool(GPU_ACTIVE),
        "device": DEVICE,
        "in_colab": bool(IN_COLAB),
    },
    "data": {
        "train_rows": int(len(train)),
        "test_rows": int(len(test)),
        "positive_rate": float(y.mean()),
        "features": E1_FEATURES,
    },
    "config": config_for_hash,
    "xgb_params_by_branch": {str(mb): native_params(mb) for mb in MAX_BIN_BRANCHES},
    "binning_preflight": {
        "n_numeric_above_256": n_above_256,
        "n_numeric_above_1024": n_above_1024,
        "resolution_ablation_is_meaningful": bool(RESOLUTION_ABLATION_IS_MEANINGFUL),
    },
    "results": {
        "partition_pooled_oof_auc": {str(mb): {str(s): partition_pooled[mb][s]
                                               for s in PARTITION_SEEDS}
                                     for mb in MAX_BIN_BRANCHES},
        "partition_mean_oof_auc": {str(mb): E1_OOF_AUC[mb] for mb in MAX_BIN_BRANCHES},
        "go_for_max_bin_1024": bool(GO_1024),
        "selected_max_bin": int(SELECTED_MAX_BIN),
    },
    "anchor": anchor_report,
    "submission_sha256": submission_hashes,
    "artifacts": sorted(path.name for path in ARTIFACT_DIR.glob("*")),
}
atomic_write_bytes(ARTIFACT_DIR / "run_manifest.json",
                   json.dumps(manifest, indent=2, default=str).encode("utf-8"))

print(json.dumps({k: manifest[k] for k in ("notebook", "config_signature", "results")},
                 indent=2, default=str))
print()
print("artifacts written to", ARTIFACT_DIR)
for path in sorted(ARTIFACT_DIR.glob("*")):
    print(f"  {path.name:<44} {path.stat().st_size / 1024:>10.1f} KB")

## 19. Interpretation

Fill this in from the printed output. The three headings exist to keep them apart.

### Measured evidence

- Pooled OOF AUC per partition for each branch, and the 15 paired fold deltas.
- Whether the `max_bin=1024` branch cleared the GO rule.
- Spearman correlation between the two branches, and between each branch and the anchor.
- Rank displacement and extreme-tail overlap against the anchor.

These are the only numbers in this notebook computed against labels or against real
prediction files. Everything else is inference.

### Assumptions

- The member's public AUC is unknown; section 15 sweeps `OOF + 0.0000` to `OOF + 0.0015`
  rather than committing. The single observed V5 offset of about +0.00106 is a prior
  from one data point.
- `rho` is unconditional Spearman on unlabeled test rows, standing in for the
  within-class correlation the binormal model requires.
- The binormal model assumes equal class-conditional variances and normal scores. Real
  blend gains are typically somewhat smaller than it predicts.
- The anchor's public AUC of 0.97128 is a displayed leaderboard value measured on
  roughly 59,260 rows, where the standard error of an AUC near 0.971 is on the order of
  0.0007.

### Unknowns

- The anchor has no aligned OOF predictions, so no blend weight can be validated against
  labels in this project. That is the single largest gap.
- Whether the OOF-to-public offset is stable across pipelines. It was measured once.
- How E1 behaves inside a multi-member stack, which is what actually matters. A member's
  pairwise correlation with the anchor is a screen, not a substitute for measuring its
  marginal contribution to a fitted stack.
- Private-leaderboard behaviour. The public split is about 20% of the test rows, and
  public-to-private movement for a fixed model is on the order of 0.0008.

### What would change the conclusion

If the section 6 pre-flight showed that no numeric feature exceeds 256 distinct values,
the resolution ablation was void from the start and no amount of GPU time would have
produced a result. In that case the honest report is "not testable on this data", not
"no effect found". The starved branch remains worth keeping or discarding purely on its
measured correlation with the anchor and V5.

## 20. How to run this on Colab

1. **Runtime → Change runtime type → T4 GPU.** Confirm with the section 8 probe; it
   trains a two-round model on CUDA rather than trusting a version string.
2. Add the Colab Secret `KAGGLE_API_TOKEN` (key icon in the left sidebar) and enable
   notebook access for it. The value is your Kaggle API key. If a token was ever visible
   in a screenshot, revoke and regenerate it on Kaggle first.
3. Copy `submission_s6e8_community_blend.csv` into
   `MyDrive/s6e8_v6_e1/anchors/`, or let section 14 prompt you to upload it.
4. Run cells 1 to 8 and **stop at the section 6 pre-flight verdict.** If it reports the
   ablation is void, decide there whether to continue.
5. Read the section 10 runtime estimate before starting section 11.
6. Run section 11. If Colab disconnects, re-run the notebook from the top; completed
   folds reload from Drive.
7. Read sections 12 to 15. Only then consider section 17.

Both submission CSVs are written to Drive and copied to `/content/` for download.